In [11]:
# =============================================================================
#  Iris Classification with PyTorch + MLflow Model Registry
# =============================================================================

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
from mlflow.tracking import MlflowClient
import pandas as pd

In [12]:
# ────────────────────────────────────────────────
#  1. Configuration & Tracking Setup
# ────────────────────────────────────────────────

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Iris_PyTorch_Classification")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [13]:
# ────────────────────────────────────────────────
#  2. Data Preparation
# ────────────────────────────────────────────────

iris = load_iris()
X = iris.data.astype(np.float32)
y = iris.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ── to PyTorch tensors ───────────────────────────────────────
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)
test_dataset  = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [14]:
# ────────────────────────────────────────────────
#  3. Model Definition
# ────────────────────────────────────────────────

class IrisNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)           # raw logits
        return x

model = IrisNet().to(DEVICE)

In [15]:
# ────────────────────────────────────────────────
#  4. Training function
# ────────────────────────────────────────────────

def train_model(model, train_loader, epochs=80, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * Xb.size(0)

        if (epoch+1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}   loss: {running_loss/len(train_loader.dataset):.4f}")

    return model

In [16]:
# ────────────────────────────────────────────────
#  5. Evaluation function
# ────────────────────────────────────────────────

@torch.no_grad()
def evaluate_model(model, test_loader):
    model.eval()
    preds = []
    trues = []

    for Xb, yb in test_loader:
        Xb = Xb.to(DEVICE)
        logits = model(Xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        preds.extend(pred)
        trues.extend(yb.numpy())

    acc       = accuracy_score(trues, preds)
    f1        = f1_score(trues, preds, average="weighted")
    precision = precision_score(trues, preds, average="weighted")
    recall    = recall_score(trues, preds, average="weighted")

    return acc, f1, precision, recall, np.array(preds)

In [17]:
# ────────────────────────────────────────────────
#  6. MLflow Logging + Model Registry
# ────────────────────────────────────────────────

with mlflow.start_run(run_name="pytorch-iris-run-002") as run:

    # ── Hyperparameters ───────────────────────────────
    epochs = 80
    lr     = 0.01
    hidden = 16

    mlflow.log_params({
        "epochs": epochs,
        "learning_rate": lr,
        "hidden_dim": hidden,
        "optimizer": "Adam",
        "criterion": "CrossEntropyLoss",
        "batch_size": 32,
        "architecture": "3-layer MLP"
    })

    # Train
    trained_model = train_model(model, train_loader, epochs=epochs, lr=lr)

    # Evaluate
    accuracy, f1, precision, recall, predictions = evaluate_model(trained_model, test_loader)

    print(f"\nTest  accuracy  = {accuracy:.4f}")
    print(f"      f1        = {f1:.4f}")
    print(f"      precision = {precision:.4f}")
    print(f"      recall    = {recall:.4f}")

    # Log metrics
    mlflow.log_metrics({
        "accuracy":  accuracy,
        "f1_score":  f1,
        "precision": precision,
        "recall":    recall
    })

    # ── Log & register PyTorch model ─────────────────────────────
    # signature = infer_signature(X_test, predictions)   # optional

    mlflow.pytorch.log_model(
        trained_model,
        artifact_path="pytorch_model",
        registered_model_name="IrisPyTorchModel",
        # signature=signature,                     # optional
        # input_example=X_test[:5]                 # optional
    )

    print(f"Run ID: {run.info.run_id}")
    print("Model registered → IrisPyTorchModel")

Epoch  20/80   loss: 0.0930
Epoch  40/80   loss: 0.0768


2026/03/14 12:31:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch  60/80   loss: 0.0618
Epoch  80/80   loss: 0.0645

Test  accuracy  = 0.9778
      f1        = 0.9778
      precision = 0.9792
      recall    = 0.9778


2026/03/14 12:31:33 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/14 12:31:33 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.7.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/14 12:31:39 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.22.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.22.1' without the local version la

Run ID: 86a7af26b01442539d0b85232d4607ac
Model registered → IrisPyTorchModel
🏃 View run pytorch-iris-run-002 at: http://127.0.0.1:5000/#/experiments/4/runs/86a7af26b01442539d0b85232d4607ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [18]:
# ────────────────────────────────────────────────
#  7. Promote latest version to Staging
# ────────────────────────────────────────────────

client = MlflowClient()

# Get latest version that is still in "None"
versions = client.search_model_versions("name='IrisPyTorchModel'")
latest_version = max(v.version for v in versions if v.current_stage == "None")

client.transition_model_version_stage(
    name="IrisPyTorchModel",
    version=latest_version,
    stage="Staging"
)

print(f"Version {latest_version} → Staging")

Version 2 → Staging


C:\Users\rasel\AppData\Local\Temp\ipykernel_10020\3469029441.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [19]:
# ────────────────────────────────────────────────
#  8. Compare versions (simple table)
# ────────────────────────────────────────────────

versions = client.search_model_versions("name='IrisPyTorchModel'")

data = []
for v in versions:
    run = client.get_run(v.run_id)
    metrics = run.data.metrics
    data.append({
        "version": v.version,
        "stage":   v.current_stage,
        "run_id":  v.run_id[:8],
        "accuracy": metrics.get("accuracy", None)
    })

df = pd.DataFrame(data)
print("\nRegistered model versions:\n")
print(df.sort_values("version", ascending=False))

best_version = df.loc[df["accuracy"].idxmax(), "version"]
best_stage   = df.loc[df["accuracy"].idxmax(), "stage"]

print(f"\nBest version so far: {best_version}  (stage: {best_stage})")


Registered model versions:

  version    stage    run_id  accuracy
0       2  Staging  86a7af26  0.977778
1       1  Staging  82c8403e  0.933333

Best version so far: 2  (stage: Staging)


In [20]:
# ────────────────────────────────────────────────
#  9. Load model from registry and predict
# ────────────────────────────────────────────────

model_uri = f"models:/IrisPyTorchModel/{best_stage}"
loaded_model = mlflow.pytorch.load_model(model_uri)

# Example inference
loaded_model.eval()
with torch.no_grad():
    sample = torch.from_numpy(X_test[:8]).to(DEVICE)
    logits = loaded_model(sample)
    probs  = torch.softmax(logits, dim=1)
    preds  = torch.argmax(probs, dim=1).cpu().numpy()

print("\nSample predictions (first 8 test samples):")
print("Predicted:", preds)
print("True     :", y_test[:8])


Sample predictions (first 8 test samples):
Predicted: [2 1 1 1 2 2 1 1]
True     : [2 1 2 1 2 2 1 1]
